# M2 centered signed CLV embeddings — Dunnhumby seed 42

기본 ID 선호공간과 N/V CLV 공간을 분리하고, 사용자 N/V 백분위를 중심화해 저·고 사용자가 반대 방향으로 개입하게 합니다. M1, 정상 사용자 배정, shuffled-user를 validation에서 비교하며 test/holdout은 열지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil, subprocess

REVIEWED_SHA = '6fa727cf3f8ef4368c8cd81911b18095bd412dc1'
repo = Path('/content/clv-m2-lightgcn-runner')
if repo.exists():
    shutil.rmtree(repo)
subprocess.run(['git', 'clone', '-q', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
%cd /content/clv-m2-lightgcn-runner
print('검토 코드 고정 완료:', actual_sha)

In [ ]:
import json, torch
from lightgcn_clv_centered_signed import (
    configure_centered_signed_dunnhumby_run,
    preflight_summary,
    run_experiment,
)

cfg = configure_centered_signed_dunnhumby_run()
assert torch.cuda.is_available(), '런타임 > 런타임 유형 변경에서 GPU를 선택하세요.'
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))

In [ ]:
result_df = run_experiment(cfg)

In [ ]:
from IPython.display import display

columns = [
    'model_id', 'role', 'gate_shape',
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50',
    'revenue@10', 'revenue@20', 'revenue@50', 'arp@10',
    'coverage@10', 'n_distinct@10', 'exposure_entropy@10',
    'eff_catalog@10', 'top10_share@10', 'top100_share@10',
    'value_alignment', 'gamma_n', 'gamma_v',
    'gate_n_std', 'gate_v_std',
]
available = [column for column in columns if column in result_df.columns]
display(result_df[available].sort_values('model_id'))
print('최종 판정:')
print(json.dumps(result_df.attrs['screening_decision'], ensure_ascii=False, indent=2))
print('결과 파일:', result_df.attrs['result_paths'])